# PCA e SVD próprias (Cap. 1)

Implementação da PCA do zero com NumPy, numa imagem sintética pequena. Depois a mesma decomposição via SVD.

Referência de leitura: `../aulas/00_teoria.md`, seções 4.2, 4.3, 4.5, 4.8 e 4.9.

Não usa dado externo. Tudo é gerado com seed fixa, então o resultado é reprodutível.

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

## 1. Imagem sintética com bandas correlacionadas

Seção 4.2: a imagem é um array `(bandas, linhas, colunas)`. Aqui são 3 bandas de 50x50.

As bandas compartilham um sinal comum (um "brilho" espacial) mais ruído próprio. Assim elas ficam correlacionadas, como bandas ópticas vizinhas.

In [ ]:
N, rows, cols = 3, 50, 50
y, x = np.mgrid[0:rows, 0:cols]
base = np.sin(x / 8.0) + np.cos(y / 11.0)            # sinal espacial comum
pesos = np.array([1.0, 0.8, 0.5])                     # quanto cada banda segue o sinal
ruido = np.array([0.15, 0.25, 0.40])                  # ruído próprio de cada banda
image = np.stack([100 + 20 * pesos[k] * base + 20 * ruido[k] * rng.standard_normal((rows, cols))
                  for k in range(N)])
print("shape:", image.shape)
print("média por banda:", image.reshape(N, -1).mean(axis=1))

## 2. Matriz de dados $G$ centrada

Seção 4.3: cada pixel é um vetor de $N$ valores. Empilhando os $m$ = linhas x colunas pixels obtemos $G$ com shape `(m, N)`.

Seção 4.8, passo 2: subtrair a média de cada banda.

In [ ]:
G = image.reshape(N, -1).T          # (m, N)
G = G - G.mean(axis=0)              # centrar por banda
m = G.shape[0]
print("G:", G.shape, "| média após centrar:", G.mean(axis=0))

## 3. Covariância $C = G^\top G/(m-1)$

Seção 4.5, covariância. A diagonal é a variância de cada banda; fora da diagonal está a redundância entre bandas.

In [ ]:
C = G.T @ G / (m - 1)
print("C =\n", C)
corr = C / np.sqrt(np.outer(np.diag(C), np.diag(C)))
print("correlação =\n", corr)

## 4. Autovalores e autovetores de $C$, do zero

Seção 4.5, autovalores. Queremos $C\mathbf{v} = \lambda\mathbf{v}$.

Para não usar caixa-preta, uso iteração de potência: multiplicar um vetor por $C$ várias vezes faz ele se alinhar à direção de maior variância. Para achar a próxima direção, o vetor é reortogonalizado (Gram-Schmidt) contra as já encontradas a cada iteração. Isso equivale à deflação, mas mantém $U^\top U = I$ mesmo com arredondamento.

Cuidados da função:

- A convergência do $k$-ésimo vetor depende da razão $\lambda_{k+1}/\lambda_k$. Perto de 1, converge devagar. Se o laço termina sem convergir, a função emite um `warning`.
- Se o resíduo fica desprezível (autovalor zero, matriz de posto deficiente), a função para e completa a base com vetores ortogonais aos já achados, com $\lambda = 0$.
- Convenção de sinal: em cada autovetor, a componente de maior módulo fica positiva. Assim o resultado não depende do vetor inicial aleatório.

In [ ]:
def fixar_sinal(U):
    """Em cada coluna, a componente de maior módulo fica positiva."""
    i = np.argmax(np.abs(U), axis=0)
    s = np.sign(U[i, np.arange(U.shape[1])])
    s[s == 0] = 1
    return U * s


def autodecomp_potencia(C, n_iter=5000, tol=1e-10, seed=0):
    """Autovalores (decrescentes) e autovetores de uma matriz simétrica positiva semidefinida."""
    C = np.asarray(C, dtype=float)
    n = C.shape[0]
    escala = np.abs(C).max()                 # escala de C, para tolerâncias relativas
    if escala == 0:                          # C = 0: qualquer base ortonormal serve
        return np.zeros(n), np.eye(n)
    r = np.random.default_rng(seed)
    lams, vecs = [], []
    for k in range(n):
        Q = np.column_stack(vecs) if vecs else np.zeros((n, 0))
        v = r.standard_normal(n)
        v -= Q @ (Q.T @ v)
        v /= np.linalg.norm(v)
        convergiu, esgotou = False, False
        for _ in range(n_iter):
            w = C @ v
            w -= Q @ (Q.T @ w)               # reortogonaliza contra os vetores já achados
            nw = np.linalg.norm(w)
            if nw <= 1e-12 * escala:         # resíduo esgotado: autovalores restantes ~ 0
                esgotou = True
                break
            lam = v @ w                      # quociente de Rayleigh
            res = np.linalg.norm(w - lam * v)
            if res <= tol * escala:
                convergiu = True
                break
            v = w / nw
        if esgotou:
            break
        if not convergiu:
            warnings.warn(f"autovetor {k + 1} não convergiu em {n_iter} iterações "
                          f"(resíduo relativo {res / escala:.1e}); autovalores próximos convergem devagar")
        lams.append(lam)
        vecs.append(v)
    # completa a base com vetores ortogonais aos já achados (autovalor zero)
    for e in np.eye(n):
        if len(vecs) == n:
            break
        Q = np.column_stack(vecs) if vecs else np.zeros((n, 0))
        u = e - Q @ (Q.T @ e)
        if np.linalg.norm(u) > 1e-8:
            vecs.append(u / np.linalg.norm(u))
            lams.append(0.0)
    lams, U = np.array(lams), np.column_stack(vecs)
    idx = np.argsort(lams)[::-1]             # ordem decrescente, como no ex01_2.py
    return lams[idx], fixar_sinal(U[:, idx])

lams, U = autodecomp_potencia(C)
print("λ =", lams)
print("U (colunas = autovetores) =\n", U)
print("max |C U - U Λ| =", np.abs(C @ U - U * lams).max())
print("U^T U =\n", U.T @ U)

## 5. Projeção nas componentes principais

Seção 4.8, passo 4: $P = GU$. Cada coluna de $P$ é uma componente principal. A covariância de $P$ deve ser diagonal, com os $\lambda$ na diagonal.

In [ ]:
P = G @ U
CP = P.T @ P / (m - 1)
print("cov(P) =\n", CP)
pcs = P.T.reshape(N, rows, cols)             # volta ao formato de imagem

## 6. Variância explicada

Fração de cada $\lambda$ na soma. A soma dos $\lambda$ é igual ao traço de $C$ (variância total).

In [ ]:
frac = lams / lams.sum()
print("traço de C:", np.trace(C), "| soma λ:", lams.sum())
for k in range(N):
    print(f"PC{k+1}: {100*frac[k]:6.2f} %   acumulado {100*frac[:k+1].sum():6.2f} %")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(10, 2.6), dpi=72)
axes[0].bar([f"PC{k+1}" for k in range(N)], 100 * frac, color="0.4")
axes[0].set_ylabel("% variância")
for k in range(N):
    axes[k + 1].imshow(pcs[k], cmap="gray")
    axes[k + 1].set_title(f"PC{k+1}")
    axes[k + 1].axis("off")
plt.tight_layout()
plt.show()

## 7. SVD de $G$

Seção 4.9. Para $m \ge N$ (caso de imagem), $G = W S V^\top$, com $W$ `(m, N)` de colunas ortonormais, $S$ diagonal com os valores singulares $s_k$ e $V$ `(N, N)` ortogonal.

Como $G^\top G = V S^2 V^\top$, as colunas de $V$ são os autovetores de $C$ e $\lambda_k = s_k^2/(m-1)$. Autovetores de autovalores distintos só são definidos a menos do sinal, então aplico a mesma convenção de sinal a $V$ antes de comparar com $U$.

In [ ]:
W, s, Vt = np.linalg.svd(G, full_matrices=False)
V = fixar_sinal(Vt.T)
print("s =", s)
print("s^2/(m-1) =", s**2 / (m - 1))
print("λ (potência) =", lams)
print("max |s^2/(m-1) - λ| =", np.abs(s**2 / (m - 1) - lams).max())
print("max |V - U| =", np.abs(V - U).max())
print("max |G V - W S| =", np.abs(G @ Vt.T - W * s).max())   # P = GV = WS

## 8. Checagem com `np.linalg.eigh`

Só para conferir a implementação própria. `eigh` devolve em ordem crescente, então inverto.

In [ ]:
lam_ref, U_ref = np.linalg.eigh(C)
lam_ref, U_ref = lam_ref[::-1], fixar_sinal(U_ref[:, ::-1])
print("max |λ_ref - λ| =", np.abs(lam_ref - lams).max())
print("max |U_ref - U| =", np.abs(U_ref - U).max())
assert np.allclose(lam_ref, lams) and np.allclose(U_ref, U, atol=1e-8)
assert np.allclose(s**2 / (m - 1), lams) and np.allclose(V, U, atol=1e-8)
print("ok: potência, eigh e SVD concordam")

## 9. Casos difíceis

Três matrizes onde a iteração de potência ingênua falha:

- $C = 0$: antes dava `nan` (divisão por norma zero).
- Posto 1, $C = \mathbf{a}\mathbf{a}^\top$: antes $U$ não saía ortonormal.
- Autovalores quase iguais (10, 1 e 0.9999): o segundo vetor converge muito devagar. Agora a função avisa. Nesse caso o autoespaço de $\{1, 0.9999\}$ é quase degenerado, então os vetores dentro dele não são bem determinados; o que importa é que $U$ continue ortonormal e os $\lambda$ fiquem perto dos verdadeiros.

In [ ]:
def checar(nome, C, lam_verdade):
    with warnings.catch_warnings(record=True) as avisos:
        warnings.simplefilter("always")
        l, Uc = autodecomp_potencia(C)
    print(f"{nome}")
    print("  λ =", l, "| verdadeiro =", np.asarray(lam_verdade))
    print("  max |U^T U - I| =", f"{np.abs(Uc.T @ Uc - np.eye(len(l))).max():.1e}",
          "| max |C U - U Λ| =", f"{np.abs(C @ Uc - Uc * l).max():.1e}",
          "| nan:", bool(np.isnan(l).any() or np.isnan(Uc).any()))
    for a in avisos:
        print("  warning:", a.message)
    assert not np.isnan(l).any() and np.allclose(Uc.T @ Uc, np.eye(len(l)))
    assert np.allclose(l, lam_verdade, atol=1e-3)

checar("C = 0", np.zeros((3, 3)), [0, 0, 0])

a = np.array([1.0, 2.0, 3.0])
checar("posto 1", np.outer(a, a), [a @ a, 0, 0])

Qr, _ = np.linalg.qr(np.random.default_rng(1).standard_normal((3, 3)))
checar("autovalores próximos", Qr @ np.diag([10, 1, 0.9999]) @ Qr.T, [10, 1, 0.9999])

## Resumo

- PCA = autodecomposição da covariância das bandas centradas.
- SVD de $G$ dá o mesmo resultado sem formar $C$: $V$ são os autovetores e $s_k^2/(m-1)$ são os autovalores.
- Na imagem sintética a PC1 é o sinal comum (brilho) e concentra quase toda a variância. PC2 e PC3 são basicamente ruído.
- A iteração de potência é didática. Para dado real, `np.linalg.eigh` ou SVD são mais robustos; a função própria serve para conferir o entendimento.
- Próximo passo: comparar com o `ex01_2.py` aplicado ao raster sintético.